In [ ]:
!pip install datasets transformers pandas


In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer


In [ ]:
# Local path for caching the dataset
cache_dir = "/content/dataset_cache"
os.makedirs(cache_dir, exist_ok=True)

# Dataset name from Hugging Face
dataset_name = "ButterChicken98/plantvillage-image-text-pairs"
dataset_path = os.path.join(cache_dir, "plant_dataset.arrow")

if os.path.exists(dataset_path):
    print("📁 Loading cached dataset...")
    dataset = Dataset.load_from_disk(cache_dir)
else:
    print("🌐 Downloading dataset from Hugging Face...")
    dataset = load_dataset(dataset_name, split="train")
    dataset.save_to_disk(cache_dir)
    print("✅ Dataset downloaded and saved to cache.")

# Display a few rows
print(dataset)
dataset_df = dataset.to_pandas()
dataset_df.head()


🌐 Downloading dataset from Hugging Face...


README.md:   0%|          | 0.00/365 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20638 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20638 [00:00<?, ? examples/s]

✅ Dataset downloaded and saved to cache.
Dataset({
    features: ['image', 'caption', 'captions'],
    num_rows: 20638
})


,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [ ]:
# Drop rows with missing caption
dataset_df = dataset_df.dropna(subset=['caption'])

# Remove duplicate entries
dataset_df = dataset_df.drop_duplicates(subset=['caption'])

# Clean text (remove unwanted whitespace)
dataset_df['caption'] = dataset_df['caption'].str.strip()

print("✅ Cleaned caption samples:")
print(dataset_df['caption'].head())


✅ Cleaned caption samples:
0         Tomato healthy
1     Tomato Late blight
3    Tomato mosaic virus
4    Pepper bell healthy
5    Potato Early blight
Name: caption, dtype: object


In [ ]:
# Check available columns
print(dataset_df.columns)


Index(['image', 'caption', 'captions'], dtype='object')


In [ ]:
# We'll use the 'caption' column as the text field
text_column = "caption"

# Drop rows with missing text
dataset_df = dataset_df.dropna(subset=[text_column])

# Remove duplicate entries
dataset_df = dataset_df.drop_duplicates(subset=[text_column])

# Clean text (remove unwanted whitespace)
dataset_df[text_column] = dataset_df[text_column].str.strip()

print("✅ Cleaned text samples:")
print(dataset_df[text_column].head())


✅ Cleaned text samples:
0         Tomato healthy
1     Tomato Late blight
3    Tomato mosaic virus
4    Pepper bell healthy
5    Potato Early blight
Name: caption, dtype: object


In [ ]:
# Load pretrained tokenizer (you can also use 'distilbert-base-uncased' or another model)
tokenizer_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

print("✅ Tokenizer loaded:", tokenizer_name)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✅ Tokenizer loaded: bert-base-uncased


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples[text_column],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Convert cleaned pandas DataFrame back to Hugging Face Dataset
dataset = Dataset.from_pandas(dataset_df)

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("✅ Tokenization complete!")
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])
tokenized_dataset.save_to_disk(os.path.join(cache_dir, "tokenized_dataset"))


Map:   0%|          | 0/15 [00:00<?, ? examples/s]

✅ Tokenization complete!


Saving the dataset (0/1 shards):   0%|          | 0/15 [00:00<?, ? examples/s]

In [ ]:
sample = tokenized_dataset[0]
print("Sample tokenized text:")
print(sample)
print("Decoded text:")
print(tokenizer.decode(sample['input_ids']))


Sample tokenized text:
{'input_ids': tensor([  101, 20856,  7965,   102,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0

In [ ]:
from transformers import DataCollatorWithPadding

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Load pretrained model suitable for sequence classification (e.g., BERT)
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Change num_labels if needed

# Create DataCollator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./model_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=True,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Replace with validation dataset if you have one
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train the model
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Assuming tokenizer is already defined above like:
# tokenizer_name = "bert-base-uncased"
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Create the data collator with tokenizer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load pretrained model with classification head for 2 labels (adjust num_labels as needed)
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Training arguments without unsupported args for older transformers versions
training_args = TrainingArguments(
    output_dir='./model_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir='./logs',
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=True,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Replace with validation dataset if any
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ValueError: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: IntervalStrategy.NO
- Save strategy: SaveStrategy.STEPS

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Assuming tokenizer is already defined above
# tokenizer_name = "bert-base-uncased"
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Create the data collator using the tokenizer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load pretrained model with classification head, adjust num_labels as per your task
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define training arguments (load_best_model_at_end disabled to fix error)
training_args = TrainingArguments(
    output_dir='./model_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir='./logs',
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=False,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Replace with a validation split if available
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-28474724.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:


Abort: 

In [ ]:
import os
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Disable wandb to prevent API key prompt
os.environ["WANDB_DISABLED"] = "true"

# Assuming tokenizer is already defined earlier
# tokenizer_name = "bert-base-uncased"
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Create data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load the pretrained model for sequence classification
model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Define training arguments without wandb integration and compatible with older transformers
training_args = TrainingArguments(
    output_dir='./model_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir='./logs',
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=False,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Replace with validation dataset if available
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Run training
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-3092182006.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Disable wandb logging to avoid API key prompt
os.environ["WANDB_DISABLED"] = "true"

# Constants
cache_dir = "/content/dataset_cache"
dataset_name = "ButterChicken98/plantvillage-image-text-pairs"

# Create local cache directory
os.makedirs(cache_dir, exist_ok=True)

# Load dataset (from cache or Hugging Face)
dataset_path = os.path.join(cache_dir, "plant_dataset.arrow")
if os.path.exists(dataset_path):
    print("📁 Loading cached dataset...")
    dataset = Dataset.load_from_disk(cache_dir)
else:
    print("🌐 Downloading dataset from Hugging Face...")
    dataset = load_dataset(dataset_name, split="train")
    dataset.save_to_disk(cache_dir)
    print("✅ Dataset downloaded and saved to cache.")

# Convert to pandas DataFrame to clean and preprocess if needed
dataset_df = dataset.to_pandas()
dataset_df.dropna(subset=["caption"], inplace=True)
dataset_df.drop_duplicates(inplace=True)

# Convert back to Hugging Face Dataset
dataset = Dataset.from_pandas(dataset_df)

# Load tokenizer
tokenizer_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["caption"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Add labels field needed by Trainer (assuming 'label' column exists and is integer class)
def add_labels(example):
    example["labels"] = example["label"]
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)

# Set format for PyTorch
tokenized_dataset.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load model for sequence classification (adjust num_labels according to your dataset)
model = AutoModelForSequenceClassification.from_pretrained(tokenizer_name, num_labels=2)

# Training arguments
training_args = TrainingArguments(
    output_dir='./model_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir='./logs',
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=False,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Optionally replace with validation dataset
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()


🌐 Downloading dataset from Hugging Face...


Saving the dataset (0/1 shards):   0%|          | 0/20638 [00:00<?, ? examples/s]

✅ Dataset downloaded and saved to cache.


TypeError: unhashable type: 'dict'

In [ ]:
import os
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Disable wandb to prevent API prompt
os.environ["WANDB_DISABLED"] = "true"

# Constants
cache_dir = "/content/dataset_cache"
dataset_name = "ButterChicken98/plantvillage-image-text-pairs"

# Create cache directory
os.makedirs(cache_dir, exist_ok=True)

# Load dataset (from cache or Hugging Face)
dataset_path = os.path.join(cache_dir, "plant_dataset.arrow")
if os.path.exists(dataset_path):
    print("📁 Loading cached dataset...")
    dataset = Dataset.load_from_disk(cache_dir)
else:
    print("🌐 Downloading dataset from Hugging Face...")
    dataset = load_dataset(dataset_name, split="train")
    dataset.save_to_disk(cache_dir)
    print("✅ Dataset downloaded and saved to cache.")

# Convert to pandas for preprocessing
dataset_df = dataset.to_pandas()
dataset_df.dropna(subset=["caption"], inplace=True)
# Drop duplicate captions to avoid unhashable errors
dataset_df.drop_duplicates(subset=["caption"], inplace=True)

# Convert back to dataset
dataset = Dataset.from_pandas(dataset_df)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples["caption"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Add 'labels' field from existing 'label' column
def add_labels(example):
    example["labels"] = example["label"]
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)

# Set dataset format for PyTorch
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# Create DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Set training arguments
training_args = TrainingArguments(
    output_dir='./model_output',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir='./logs',
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=False,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # Replace with validation split if available
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Start training
trainer.train()


🌐 Downloading dataset from Hugging Face...


Saving the dataset (0/1 shards):   0%|          | 0/20638 [00:00<?, ? examples/s]

✅ Dataset downloaded and saved to cache.


Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

KeyError: 'label'

In [ ]:
print(tokenized_dataset.column_names)


['image', 'caption', 'captions', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# ✅ Disable wandb logging to avoid API key prompt
os.environ["WANDB_DISABLED"] = "true"

# === CONFIG ===
cache_dir = "/content/dataset_cache"
dataset_name = "ButterChicken98/plantvillage-image-text-pairs"

# === LOAD DATASET ===
os.makedirs(cache_dir, exist_ok=True)
dataset_path = os.path.join(cache_dir, "plant_dataset.arrow")

if os.path.exists(dataset_path):
    print("📁 Loading cached dataset...")
    dataset = Dataset.load_from_disk(cache_dir)
else:
    print("🌐 Downloading dataset from Hugging Face...")
    dataset = load_dataset(dataset_name, split="train")
    dataset.save_to_disk(cache_dir)
    print("✅ Dataset downloaded and saved to cache.")

# === CLEAN DATA ===
dataset_df = dataset.to_pandas()
dataset_df.dropna(subset=["caption"], inplace=True)
dataset_df.drop_duplicates(subset=["caption"], inplace=True)

# ✅ If dataset has no 'label' column, create dummy labels
if "label" not in dataset_df.columns:
    dataset_df["label"] = 0  # all samples have label 0

# Convert back to Hugging Face Dataset
dataset = Dataset.from_pandas(dataset_df)

# === TOKENIZATION ===
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["caption"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# ✅ Add labels (now guaranteed to exist)
def add_labels(example):
    example["labels"] = example["label"]
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)

# === FORMAT FOR TRAINING ===
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# === MODEL & TRAINER SETUP ===
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./model_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=500,
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,  # You can replace with a validation split later
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# === TRAIN ===
print("🚀 Starting training...")
trainer.train()

print("✅ Training completed successfully!")
print("Columns in tokenized dataset:", tokenized_dataset.column_names)


🌐 Downloading dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/365 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20638 [00:00<?, ? examples/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-286723713.py:87: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


✅ Training completed successfully!
Columns in tokenized dataset: ['image', 'caption', 'captions', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [ ]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate
import numpy as np
import os

# Disable WandB
os.environ["WANDB_DISABLED"] = "true"

# === Load dataset from Hugging Face Parquet ===
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

# Drop image column
df = df.drop(columns=["image"])
print(" Dataset loaded:", df.shape)

# Explode the captions list
df = df.explode("captions").reset_index(drop=True)
print(" After exploding:", df.shape)

# Encode labels (caption as label)
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["caption"])

# Split train/test (80/20)
df_train, df_test = train_test_split(df, train_size=0.8, random_state=42)

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

# === Tokenization ===
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["captions"], truncation=True)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

# === Data Collator and Model ===
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
num_labels = len(label_encoder.classes_)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# === Evaluation metric ===
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# === Training arguments ===
training_args = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,           # You can increase this for more training
    learning_rate=5e-5,
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    weight_decay=0.01,
    report_to="none"
)

# === Trainer ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# === Train ===
print("🚀 Starting training...")
trainer.train()

# === Evaluate ===
results = trainer.evaluate()
print("📊 Final evaluation results:", results)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Dataset loaded: (20638, 2)
✅ After exploding: (82552, 2)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/tmp/ipython-input-2275110711.py:79: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting training...


Step,Training Loss
8256,0.022200
16512,0.000000


📊 Final evaluation results: {'eval_loss': 9.681286172735781e-08, 'eval_accuracy': 1.0, 'eval_runtime': 17.611, 'eval_samples_per_second': 937.542, 'eval_steps_per_second': 117.2, 'epoch': 2.0}


In [ ]:
import torch

# Detect device automatically
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_text(text):
    # Tokenize and move tensors to same device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    # Disable gradient calculation (for faster inference)
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=-1).item()

    # Decode numeric label → text label
    predicted_label = label_encoder.inverse_transform([predicted_class_id])[0]

    print(f"\n🪴 Input text: {text}")
    print(f"🔮 Predicted label: {predicted_label}")

# Try with some examples
predict_text("a healthy tomato leaf with green color")
predict_text("tomato leaf with yellow spots and fungal infection")
predict_text("potato leaf showing early blight symptoms")



🪴 Input text: a healthy tomato leaf with green color
🔮 Predicted label: Tomato healthy

🪴 Input text: tomato leaf with yellow spots and fungal infection
🔮 Predicted label: Tomato Bacterial spot

🪴 Input text: potato leaf showing early blight symptoms
🔮 Predicted label: Potato Early blight


In [ ]:
# If you have access to the trainer and want to save everything (model, tokenizer, config):
trainer.save_model("./saved_model")

# Or to save just the model in Hugging Face format:
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")


('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/vocab.txt',
 './saved_model/added_tokens.json',
 './saved_model/tokenizer.json')